# Distribution Guide — Choose the Right Probability Shape

> **Choosing the right distribution can change your risk estimate by 10-30%.**

---

### Quick reference

```
What are you analyzing?
├── Conversion rate or percentage (0-100%)?  →  ✅ Use BETA
│     Best for success rates and conversion rates (bounded between 0% and 100%)
│     α / β derived from BetaParameters.from_mean_uncertainty(mean, uncertainty)
├── Business Value, adoption, or usage (unbounded)?  →  ✅ Use LOGNORMAL
│     Best for business value, deal sizes, sprint overruns — right-skewed, always > 0
└── Quick estimate, low uncertainty (<30%)?   →  ✅ Use NORMAL
      Quick approximation for symmetric, low-risk estimates
      ⚠ Auto-switched to Lognormal by the FHS model when uncertainty ≥ 0.30
```

> **FHS model auto-selection rule:** `distribution="normal"` (default) is automatically upgraded to `"lognormal"` when `uncertainty ≥ 0.30` to prevent clipping bias. Set `distribution="beta"` explicitly for conversion rates.

---
*Previous: [02 — Blockchain Case Study](../02-blockchain-case-study.ipynb)*

When uncertainty is high (≥ 0.3), the system switches from a Normal to a Lognormal model to avoid negative sampled values. This threshold is conservative and intended to prevent unrealistic negative outcomes.


📌 **Key Takeaway** — Choosing the wrong probability model can silently shift your risk estimate by 10–30 %. The distribution shape matters as much as the input numbers.

---

## Where This Fits In The Learning Path

This tutorial does **not** replace [01 Getting Started](../01-getting-started.ipynb) — it extends it.

- [01 Getting Started](../01-getting-started.ipynb): quick decision workflow for one feature (define → simulate → decide).
- This notebook (T01): model-quality workflow (choose distribution → validate risk metrics → avoid bias).

### 60-second bridge from Getting Started

Before continuing here, make sure you can already do these 3 steps from [01 Getting Started](../01-getting-started.ipynb):
1. Define one feature (`expected_users`, `conversion_rate`, `uncertainty`, `business_value_per_conversion`, `development_cost`).
2. Run one simulation with the application service.
3. Read `Expected`, `Business Value Floor`, and `P95` from the result card.

If yes, this notebook is your next step to improve **model selection quality** and make business value floor / CVaR numbers decision-grade.

In [ ]:
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show

setup = notebook_setup("blockchain", show_status=False)
scenario = setup.scenario

if scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")

show.info("✅ <b>Environment ready</b>")

---

## Three Distributions, One Conversion Rate

The same conversion rate (8%) modeled three different ways. **Notice how different the risk estimates are!**

In [ ]:
mean_rate = 0.08
summary = show.distribution_shape_comparison(
    mean_rate=mean_rate, n_samples=10_000, seed=42
)

show.info(
    f"<b>Key insight:</b> The business value floor differs by up to <b>{summary['var_max_diff_pct']:.0f}%</b> "
    f"depending on which distribution you choose.<br>"
    f'<span style="font-size:13px;">Normal: {summary["var_normal"]:.4f} &nbsp;|&nbsp; '
    f"Lognormal: {summary['var_lognormal']:.4f} &nbsp;|&nbsp; Beta: {summary['var_beta']:.4f}</span>"
)

**Chart guide — Three distribution models for the same conversion rate**

All three histograms model the same underlying conversion rate (8 %) using 10,000 samples. The vertical lines mark the mean (solid dark), the 5th percentile (dashed red), and the 95th percentile (dashed green).

- **Normal:** Symmetric bell shape. Tails extend equally in both directions, including into negative values, which must be clipped to zero. This clipping introduces a small upward bias in the mean.
- **Lognormal:** Asymmetric with a longer right tail. All values are positive by definition. The median is below the mean; large values are possible but rare.
- **Beta:** Bounded strictly between 0 and 1 by definition — suitable for rates and proportions. The shape depends on the parameters α and β, which correspond to observed successes and failures in historical data.

> The **Range** badge shows `P95 − P5`, the width of the central 90 % interval. A wider range means more uncertainty in the estimate.


---

## 1. Normal Distribution — The Quick Estimate

| ✅ Use when | ❌ Avoid when |
|------------|-------------|
| Quick estimates with low uncertainty | Uncertainty > 50% |
| Symmetric data | Right-skewed data (business value, usage) |
| Limited historical data | You need bounded [0,1] values |

In [ ]:
summary = show.normal_distribution_deep_dive(
    mean_rate=0.08,
    high_uncertainty=0.5,
    n_samples=10_000,
    seed=42,
)

show.info(
    f"⚠️ <b>{summary['clipped_pct']:.1f}%</b> of normal samples were negative and had to be clipped to 0. "
    f"This introduces a small upward bias in the mean."
)

**Chart guide — Normal distribution properties**

- **Left: Clipping bias.** The raw normal distribution (red) can produce negative values, which are impossible for a conversion rate. Clipping at zero (blue) removes these and piles up probability mass at the boundary. The share of clipped samples is shown below the chart. This bias grows with uncertainty.
- **Right: Uncertainty levels.** Three overlapping histograms show the same mean (8 %) with different standard deviations, determined by the uncertainty parameter. A low uncertainty parameter produces a narrow peak; a high value spreads the distribution widely.


---

## 2. Lognormal Distribution — For Right-Skewed Data

| ✅ Use when | ❌ Avoid when |
|------------|-------------|
| Business Value per customer | Bounded [0,1] rates |
| Market adoption rates | Symmetric data |
| Feature usage frequency | Need simplicity (use Normal instead) |
| **Sprint cost overruns** (Notebook 06) | |

> **Why?** Most product metrics follow the **80/20 rule**: few power users, many casual ones. The same applies to sprint overruns — small delays are common, catastrophic ones are rare but skew the cost distribution right.

> **FHS auto-switch:** The simulator automatically selects Lognormal over Normal when `uncertainty ≥ 0.30`. Sprint delivery cost is modelled with Lognormal by default in Notebook 06.

In [ ]:
summary = show.lognormal_distribution_deep_dive(
    normal_mean=1_000.0,
    normal_std=500.0,
    lognormal_scale=800.0,
    lognormal_sigma=0.5,
    n_samples=10_000,
    seed=42,
)

show.info(
    f"📊 <b>Skewness:</b> Normal = {summary['skew_normal']:.2f} (symmetric) vs. "
    f"Lognormal = {summary['skew_lognormal']:.2f} (right tail). "
    f'The lognormal captures the "long tail" of high-business value customers that Normal misses.'
)

**Chart guide — Lognormal distribution properties**

- **Left: Business Value distributions.** A Normal and a Lognormal distribution are shown for the same approximate mean business value. The lognormal has a longer right tail, which reflects the "power law" pattern common in customer business value: many customers generate modest business value, while a small number generate disproportionately large amounts.
- **Right: Box plots.** Each box shows the interquartile range (25th to 75th percentile); the line inside is the median; whiskers extend to ±1.5 × IQR; points beyond are outliers. The lognormal box is asymmetric and has more high-value outliers, reflecting its right skew.


> 💡 **Why is the mean higher than the median?** With a lognormal distribution, the average (mean) is always higher than the median because rare high outcomes pull the mean up. This is realistic for business value — a few big wins drive average performance above typical performance.

---

## 3. Beta Distribution — For Rates and Percentages

| ✅ Use when | ❌ Avoid when |
|------------|-------------|
| Conversion rates (0-100%) | Unbounded data (business value) |
| Success/failure rates | Only one data point (use Normal) |
| Historical data available | |

> **Why?** Beta is **designed** for values between 0 and 1. No clipping needed!

#### When does clipping become a problem with Normal distribution?

When you use `distribution="normal"` for a conversion rate, the simulator clips negative samples to 0. This is harmless for typical rates — but becomes a material bias when:

| Condition | Clipping share | Impact |
|-----------|---------------|--------|
| `conversion_rate ≥ 10 %`, any uncertainty | < 1 % | Negligible |
| `conversion_rate = 5 %`, `uncertainty = 30 %` | ~2 % | Minor |
| `conversion_rate = 5 %`, `uncertainty ≥ 40 %` | **> 5 %** | **Material — use Beta** |
| `conversion_rate < 5 %`, `uncertainty ≥ 30 %` | **> 10 %** | **Severe — use Beta** |

**Rule of thumb:** If `conversion_rate / uncertainty < 0.15`, switch to Beta or Lognormal. FHS will log a warning automatically when > 1 % of samples are clipped.

In [ ]:
# ── Beta Distribution: Effect of Sample Size ─────────────────────────
show.beta_sample_size_effect(n_samples=10_000, seed=scenario.seed)

show.info(
    "<b>More data = narrower distribution = more confidence.</b> "
    "After 3,000 trials the 95% confidence interval is extremely tight."
)

**Chart guide — Beta distribution and sample size**

The three panels show the same underlying conversion rate (≈ 8 %) modelled with different amounts of historical data. The Beta($\alpha$, $\beta$) distribution has mean $\alpha / (\alpha + \beta)$. As $\alpha + \beta$ (the total number of observed trials) increases, the distribution narrows.

- **Low confidence (25 trials):** Wide spread; the 95 % credible interval spans a large range. Small samples leave much uncertainty.
- **Medium confidence (200 trials):** The distribution narrows considerably. Decisions can be made with more confidence.
- **High confidence (3 000 trials):** Very narrow peak. After sufficient trials, the conversion rate is known precisely.

> The shaded area marks the 95 % credible interval [`P2.5`, `P97.5`] — the range containing 95 % of the simulated values.


---

### Beta vs Normal — Box Plot Comparison

The Normal distribution clips negative samples to 0, introducing an upward bias. The Beta distribution is bounded by design. Both are parameterised from the same inputs (mean, uncertainty) — the chart makes the difference visible.

> **FHS auto-selection:** The simulator switches from `"normal"` to `"lognormal"` automatically when `uncertainty ≥ 0.30`. For conversion rates, pass `distribution="beta"` explicitly to use `BetaParameters.from_mean_uncertainty()` internally.

In [ ]:
# ── Beta vs Normal: Box Plot Comparison ───────────────────────────────────────
summary = show.beta_vs_normal_boxplot(
    mean_rate=0.08,
    uncertainty=0.40,
    n_samples=10_000,
    seed=scenario.seed,
)

show.info(
    f"⚠️ Normal generates <b>{summary['clipped_pct']:.1f}%</b> negative samples that must be clipped "
    f"to 0 (uncertainty=40%). "
    f"Beta avoids this entirely — it is bounded [0, 1] by design.<br>"
    f"<span style='font-size:13px;'>"
    f"α = {summary['alpha']:.2f}, β = {summary['beta']:.2f} — derived via "
    f"<code>BetaParameters.from_mean_uncertainty(mean=0.08, uncertainty=0.40)</code>, "
    f"the same formula the FHS simulator uses internally.</span>"
)

**Chart guide — Beta vs Normal box plot comparison**

Both panels model the same conversion rate (8 %) with 40 % relative uncertainty — a level that triggers the FHS auto-switch from Normal to Lognormal (`uncertainty ≥ 0.30`).

- **Left: Histogram.** The Normal distribution (blue) shows a small pile-up at 0 % where negative samples have been clipped. The Beta distribution (green) has no such artefact — it is naturally bounded between 0 and 1. At 40 % uncertainty, Normal clips a material share of samples, creating an upward bias in the mean.
- **Right: Box plot.** The dashed line marks the true mean (8 %). The box shows the interquartile range (25th–75th percentile); whiskers extend to ±1.5 × IQR; circles are outliers. The Beta box is symmetric and unbiased. The Normal box may show a slightly elevated median and compressed lower whisker due to clipping.

> The Beta parameters α and β are derived by `BetaParameters.from_mean_uncertainty(mean=0.08, uncertainty=0.40)` — **the same formula used internally by the FHS simulator** when `distribution="beta"` is requested. This ensures the tutorial distributes identically to the production model.

---

## Distribution Choice and Risk Metrics (Floor / CVaR)

Choosing the wrong distribution doesn't just change the histogram shape — it shifts the **business value floor** and **CVaR 95%** estimates that drive every notebook in this series.

| Metric | Definition | Notebook |
|--------|-----------|----------|
| **Business Value Floor 95** | 5th percentile — the floor 95 % of scenarios exceed | 01, 02, 03, 04 |
| **CVaR 95%** | Average of the worst 5 % of scenarios | 01, 02, 05, 06 |
| **Break-even probability** | Share of scenarios with positive profit | 06 |
| **Loss at Risk 95% (LaR)** | 95th percentile of the *loss* distribution | 06 |

> **Business value floor = €0 edge case:** When feature `likelihood_of_non_delivery > 5%` (e.g., H3 in the blockchain study with LLP = 80 %), more than 5 % of scenarios deliver €0 business value. The business value floor is then structurally zero — use **Expected Value** and **break-even probability** as primary metrics instead (Notebooks 05–06).

The cell below computes all three distribution models for the same feature and compares the resulting risk metrics side by side.

In [ ]:
# ── Distribution Choice: Impact on VaR 95% and CVaR 95% ──────────────────────
summary = show.distribution_risk_metric_comparison(
    mean_rate=0.08,
    uncertainty=0.40,
    n_users=5_000,
    business_value_per_conversion=150.0,
    n_samples=50_000,
    seed=scenario.seed,
)

show.info(
    f"<b>Business Value Floor comparison (same feature, 40% uncertainty):</b> "
    f"Normal €{summary['normal_var_95'] / 1000:.1f}k &nbsp;|&nbsp; "
    f"Lognormal €{summary['lognormal_var_95'] / 1000:.1f}k &nbsp;|&nbsp; "
    f"Beta €{summary['beta_var_95'] / 1000:.1f}k. "
    f"Gap = <b>{summary['max_diff_pct_normal_vs_beta']:.0f}%</b> between Normal and Beta — "
    f"this directly shifts the budget floor used in Notebooks 02–04."
)

**Chart guide — Distribution impact on Business Value Floor and CVaR 95%**

All three panels model the same feature with identical inputs: 5,000 users, 8 % conversion rate, €150 business value per conversion, 40 % uncertainty. The only difference is the probability model.

- **Normal (clipped):** Negative samples must be clipped to zero. This piles probability mass at €0, pulling the tail down and reducing VaR 95%.
- **Lognormal:** No clipping needed. The right tail extends further, reflecting the 80/20 distribution of customer outcomes. VaR 95% is typically above the Normal model.
- **Beta:** Strictly bounded between 0 and 1 by construction. No clipping artefact. VaR 95% reflects the actual tail of the bounded conversion rate.

> The percentage gap shown in the info panel illustrates the **10–30 % risk-estimate shift** introduced by distribution choice. This is why the FHS simulator automatically switches from Normal to Lognormal when `uncertainty ≥ 0.30`. For conversion rates, use **Beta** to eliminate clipping bias entirely.

---

## Distribution decision guide

In [ ]:
_beta_card = {
    "title": "Beta",
    "color": COLORS.secondary,
    "description": "For <b>rates and percentages</b>: conversion rate, adoption rate, churn rate.",
    "bullets": [
        "Bounded between 0 and 1 — no clipping needed",
        "Shape narrows as sample size grows",
    ],
    "rule": "Metric can only be 0%–100%? Use Beta.",
}

In [ ]:
_lognormal_card = {
    "title": "Lognormal",
    "color": COLORS.accent,
    "description": "For <b>business value, deal sizes, adoption counts</b> — values that skew right.",
    "bullets": [
        "Never goes negative",
        "Captures the long tail of outlier outcomes",
    ],
    "rule": "Most values cluster low but rare highs are possible? Use Lognormal.",
}
_normal_card = {
    "title": "Normal",
    "color": COLORS.primary,
    "description": "For <b>quick estimates</b> when data is symmetric and uncertainty is low (&lt; 50%).",
    "bullets": [
        "Simple and fast to reason about",
        "Clipped to [0, 1] for rate inputs",
    ],
    "rule": "Default choice for prototyping. Switch when results look wrong.",
}

In [ ]:
# ── Distribution Decision Guide ──────────────────────────────────────
show.grid(
    [_beta_card, _lognormal_card, _normal_card],
    title="Which distribution should I use?",
    footer=(
        "<b>Quick reference:</b> "
        "Conversion rate &rarr; <b>Beta</b> &nbsp;|&nbsp; "
        "Business Value / usage &rarr; <b>Lognormal</b> &nbsp;|&nbsp; "
        "Rough estimate &rarr; <b>Normal</b>"
    ),
)

**Chart guide — Distribution decision cards**

Each card summarises one probability model: when to use it, its key properties, and the deciding rule. Use this as a quick reference before configuring a `Feature` object or interpreting simulation output.

- **Beta** — for rates and percentages (conversion rate, churn rate, acceptance rate). Bounded [0, 1] with no clipping. Required when `conversion_rate / uncertainty < 0.15`.
- **Lognormal** — for business value, deal sizes, market adoption, and sprint overruns. Always positive, right-skewed. Selected automatically when `uncertainty ≥ 0.30`.
- **Normal** — default for prototyping with low uncertainty. Symmetric bell curve; clips negative values, so use only when `uncertainty < 0.30` and the metric is not a rate.

> **Connection to Notebooks 01–06:** VaR 95% (floor), CVaR 95% (tail average), and break-even probability all depend on the distribution shape chosen here. Notebook 02 explains the 0.30 threshold; Notebooks 05–06 show the VaR = €0 edge case when delivery probability is low.

---

## Interactive Feature Risk Profile (Plotly)

Plug in your own parameters to explore how distribution choice affects the simulated risk profile.

In [ ]:
from fhs import Feature
from fhs.application import BlockchainCaseStudyService
from fhs.presentation.notebook import plot_risk_profile

interactive_feature = Feature(
    name="Premium Subscription",
    expected_users=5_000,
    conversion_rate=0.20,
    uncertainty=0.25,
    business_value_per_conversion=150.0,
    development_cost=40_000,
)

case = BlockchainCaseStudyService(seed=scenario.seed, scenarios=scenario.scenarios)
sim_results = case.simulate_year1({"T1": interactive_feature})

In [ ]:
# ── Interactive Single Feature ─────────────────────────────────────
result = sim_results["T1"].result
plot_risk_profile(result);

---

## Summary

1. **Distribution choice matters** — it shifts business value floor and CVaR 95% estimates by 10–30%
2. **Conversion rates** (bounded 0–100%) → always use **Beta** → derived via `BetaParameters.from_mean_uncertainty()`
3. **Business Value / adoption / sprint overruns** (right-skewed) → use **Lognormal**
4. **Quick estimates** with `uncertainty < 0.30` → **Normal** is a reasonable start
5. **FHS auto-selection:** `uncertainty ≥ 0.30` triggers automatic switch to Lognormal — no manual intervention needed
6. **Have historical conversion data?** → fit Beta from success/failure counts: `alpha = successes`, `beta = failures`
7. **VaR = €0 edge case:** when `likelihood_of_non_delivery > 5%`, VaR 95% is structurally €0 — use **Expected Value** and **break-even probability** instead (Notebooks 05–06)
8. **Wrong distribution = wrong budget floor** — validate with the comparison charts above before committing to a business case

---

## Notebook Navigation

| # | Notebook | Role in learning path |
|:-:|----------|-----------------------|
| 01 | [Getting Started](../01-getting-started.ipynb) | First entry point: one feature, one simulation, one decision |
| T01 | **Distribution Guide** | Second step: choose the right statistical model and validate VaR/CVaR |
| 02 | [Blockchain Case Study](../02-blockchain-case-study.ipynb) | Apply the same logic to three hypotheses and portfolio context |
| 03 | [Financing the Roadmap](../03-blockchain-case-study-capital-budgeting.ipynb) | Translate simulation outputs into ROI, NPV, IRR |
| 04 | [Portfolio Advisor](../04-blockchain-case-study-advisor.ipynb) | Optimize feature mix under budget using `var_floor` |
| 05 | [Risk Layers](../05-blockchain-case-study-risk.ipynb) | Add delivery, market, component, and global risk layers |
| 06 | [Development Risk](../06-blockchain-case-study-development-risk.ipynb) | Add sprint overrun risk, break-even probability, and Loss at Risk |
| A01 | [Portfolio Advisor](../advanced/01-portfolio-advisor.ipynb) | Advanced ranking and solver comparison |
| A02 | [Portfolio Risk Dashboard](../advanced/02-portfolio-risk-dashboard.ipynb) | Advanced risk-layer dashboard and stress-path analysis |


---

## Key Takeaway for Decision Makers

> **The choice of distribution changes your risk estimates by 10–30%.** For high-stakes decisions (>€100k), use **Beta** for conversion rates and **Lognormal** for business value and sprint overruns. For quick estimates with `uncertainty < 0.30`, **Normal** is sufficient — and the FHS simulator upgrades it automatically when uncertainty is higher.

> **When VaR = €0:** If a feature's likelihood of non-delivery exceeds 5%, VaR 95% is always €0 by design. Switch to **Expected Value**, **break-even probability**, and **Loss at Risk 95%** as your primary metrics (see Notebooks 05–06).